# CSIT 598: Assignment 3
### Neeraj Kumar Singh
### 50161746

## Objective
Implement and evaluate deep learning models on MNIST and compare them with the best model from Assignment 1 (XGBoost: accuracy 0.9695, macro F1 0.9693, train 192.5334 s, predict 0.0869 s).

## Dataset, Frameworks, and Metrics
Work with the MNIST dataset (handwritten digits), same to Assignment 1. To build deep
learning algorithms, you can use either TensorFlow or PyTorch.

## Required Metrics
- Test accuracy
- Running time

In [1]:
# Core setup, data loading, and shared utilities
import time
import random
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

BATCH_SIZE = 128
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples: {len(test_dataset)}')


def evaluate_model(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in data_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / total, correct / total


def train_model(model, train_loader, test_loader, optimizer, criterion, device, num_epochs=5):
    model = model.to(device)
    best_state = deepcopy(model.state_dict())
    best_acc = 0.0

    start_time = time.perf_counter()

    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        running_total = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)
            preds = torch.argmax(logits, dim=1)
            running_correct += (preds == y).sum().item()
            running_total += y.size(0)

        train_loss = running_loss / running_total
        train_acc = running_correct / running_total
        test_loss, test_acc = evaluate_model(model, test_loader, criterion, device)

        if test_acc > best_acc:
            best_acc = test_acc
            best_state = deepcopy(model.state_dict())

        print(
            f'Epoch {epoch:02d}/{num_epochs} | '
            f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
            f'test_loss={test_loss:.4f} test_acc={test_acc:.4f}'
        )

    elapsed = time.perf_counter() - start_time
    model.load_state_dict(best_state)
    final_test_loss, final_test_acc = evaluate_model(model, test_loader, criterion, device)

    return {
        'model': model,
        'test_loss': final_test_loss,
        'test_accuracy': final_test_acc,
        'running_time_sec': elapsed,
    }


class MetricsLogger:
    def __init__(self):
        self.rows = []

    def add(self, task, model_name, accuracy, running_time_sec, notes=''):
        self.rows.append({
            'task': task,
            'model': model_name,
            'test_accuracy': accuracy,
            'running_time_sec': running_time_sec,
            'notes': notes,
        })

    def to_dataframe(self):
        import pandas as pd
        return pd.DataFrame(self.rows)


class MLP(nn.Module):
    def __init__(self, hidden_dims, activation='relu'):
        super().__init__()

        if activation.lower() == 'relu':
            act_layer = nn.ReLU
        elif activation.lower() == 'sigmoid':
            act_layer = nn.Sigmoid
        else:
            raise ValueError("activation must be 'relu' or 'sigmoid'")

        layers = []
        in_dim = 28 * 28
        for h in hidden_dims:
            layers.append(nn.Linear(in_dim, h))
            layers.append(act_layer())
            in_dim = h
        layers.append(nn.Linear(in_dim, 10))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)


class CNN4Layer(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


class LeNet5MNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5),
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2),
            nn.Conv2d(6, 16, kernel_size=5),
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 4 * 4, 120),
            nn.Tanh(),
            nn.Linear(120, 84),
            nn.Tanh(),
            nn.Linear(84, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def build_optimizer(model, optimizer_name='adam', lr=1e-3):
    name = optimizer_name.lower()
    if name == 'adam':
        return optim.Adam(model.parameters(), lr=lr)
    if name == 'sgd':
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    raise ValueError("optimizer_name must be 'adam' or 'sgd'")


metrics = MetricsLogger()
criterion = nn.CrossEntropyLoss()

Using device: cpu
Train samples: 60000
Test samples: 10000


## Task A: Multi-Layer Perceptron


In [2]:
# Task A container
results_a = {}
print('Task A started: Multi-Layer Perceptron experiments')

Task A started: Multi-Layer Perceptron experiments


### A.1
Implement a three-layer perceptron (using ReLU as activation functions, and Adam as optimizer) and evaluate its performance.

In [3]:
# A.1: 3-layer MLP, ReLU + Adam
model_a1 = MLP(hidden_dims=[256, 128], activation='relu')
opt_a1 = build_optimizer(model_a1, optimizer_name='adam', lr=LEARNING_RATE)
results_a['A1_3layer_relu_adam'] = train_model(
    model_a1, train_loader, test_loader, opt_a1, criterion, device, num_epochs=NUM_EPOCHS
)
metrics.add(
    'A',
    'A1_3layer_relu_adam',
    results_a['A1_3layer_relu_adam']['test_accuracy'],
    results_a['A1_3layer_relu_adam']['running_time_sec'],
)
print(results_a['A1_3layer_relu_adam'])

Epoch 01/5 | train_loss=0.2716 train_acc=0.9199 | test_loss=0.1194 test_acc=0.9634
Epoch 02/5 | train_loss=0.1032 train_acc=0.9682 | test_loss=0.0813 test_acc=0.9740
Epoch 03/5 | train_loss=0.0691 train_acc=0.9784 | test_loss=0.0764 test_acc=0.9755
Epoch 04/5 | train_loss=0.0500 train_acc=0.9843 | test_loss=0.0677 test_acc=0.9785
Epoch 05/5 | train_loss=0.0404 train_acc=0.9866 | test_loss=0.0812 test_acc=0.9755
{'model': MLP(
  (net): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
), 'test_loss': 0.06770999330896885, 'test_accuracy': 0.9785, 'running_time_sec': 10.610418708001816}


### A.2
Change the activation function to Sigmoid, and evaluate the performance

In [4]:
# A.2: 3-layer MLP, Sigmoid + Adam
model_a2 = MLP(hidden_dims=[256, 128], activation='sigmoid')
opt_a2 = build_optimizer(model_a2, optimizer_name='adam', lr=LEARNING_RATE)
results_a['A2_3layer_sigmoid_adam'] = train_model(
    model_a2, train_loader, test_loader, opt_a2, criterion, device, num_epochs=NUM_EPOCHS
)
metrics.add(
    'A',
    'A2_3layer_sigmoid_adam',
    results_a['A2_3layer_sigmoid_adam']['test_accuracy'],
    results_a['A2_3layer_sigmoid_adam']['running_time_sec'],
)
print(results_a['A2_3layer_sigmoid_adam'])

Epoch 01/5 | train_loss=0.5492 train_acc=0.8621 | test_loss=0.2098 test_acc=0.9404
Epoch 02/5 | train_loss=0.1664 train_acc=0.9529 | test_loss=0.1474 test_acc=0.9553
Epoch 03/5 | train_loss=0.1070 train_acc=0.9692 | test_loss=0.1185 test_acc=0.9629
Epoch 04/5 | train_loss=0.0759 train_acc=0.9783 | test_loss=0.0877 test_acc=0.9728
Epoch 05/5 | train_loss=0.0557 train_acc=0.9841 | test_loss=0.0876 test_acc=0.9748
{'model': MLP(
  (net): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): Sigmoid()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): Sigmoid()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
), 'test_loss': 0.08757915959358216, 'test_accuracy': 0.9748, 'running_time_sec': 10.881140500001493}


### A.3
Change the optimizer from Adam to stochastic gradient descent (SGD) and evaluate its performance (while keeping activation as ReLU)

In [5]:
# A.3: 3-layer MLP, ReLU + SGD
model_a3 = MLP(hidden_dims=[256, 128], activation='relu')
opt_a3 = build_optimizer(model_a3, optimizer_name='sgd', lr=LEARNING_RATE)
results_a['A3_3layer_relu_sgd'] = train_model(
    model_a3, train_loader, test_loader, opt_a3, criterion, device, num_epochs=NUM_EPOCHS
)
metrics.add(
    'A',
    'A3_3layer_relu_sgd',
    results_a['A3_3layer_relu_sgd']['test_accuracy'],
    results_a['A3_3layer_relu_sgd']['running_time_sec'],
)
print(results_a['A3_3layer_relu_sgd'])

Epoch 01/5 | train_loss=1.2479 train_acc=0.6953 | test_loss=0.4825 test_acc=0.8734
Epoch 02/5 | train_loss=0.4058 train_acc=0.8885 | test_loss=0.3366 test_acc=0.9030
Epoch 03/5 | train_loss=0.3264 train_acc=0.9054 | test_loss=0.2928 test_acc=0.9150
Epoch 04/5 | train_loss=0.2895 train_acc=0.9159 | test_loss=0.2700 test_acc=0.9228
Epoch 05/5 | train_loss=0.2630 train_acc=0.9238 | test_loss=0.2434 test_acc=0.9309
{'model': MLP(
  (net): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
), 'test_loss': 0.24343447051048278, 'test_accuracy': 0.9309, 'running_time_sec': 10.104262666998693}


### A.4
Now implement a 5-layer perceptron using activation and optimizer that give the best performance according to A.1-A.3

In [6]:
# A.4: 5-layer MLP using best activation/optimizer from A.1-A.3
best_a13_key = max(
    ['A1_3layer_relu_adam', 'A2_3layer_sigmoid_adam', 'A3_3layer_relu_sgd'],
    key=lambda k: results_a[k]['test_accuracy'],
)
print('Best among A1-A3:', best_a13_key)

best_activation = 'sigmoid' if 'sigmoid' in best_a13_key else 'relu'
best_optimizer = 'sgd' if '_sgd' in best_a13_key else 'adam'

model_a4 = MLP(hidden_dims=[512, 256, 128, 64], activation=best_activation)
opt_a4 = build_optimizer(model_a4, optimizer_name=best_optimizer, lr=LEARNING_RATE)
results_a['A4_5layer_best_act_opt'] = train_model(
    model_a4, train_loader, test_loader, opt_a4, criterion, device, num_epochs=NUM_EPOCHS
)
metrics.add(
    'A',
    'A4_5layer_best_act_opt',
    results_a['A4_5layer_best_act_opt']['test_accuracy'],
    results_a['A4_5layer_best_act_opt']['running_time_sec'],
    notes=f'activation={best_activation}, optimizer={best_optimizer}',
)
print(results_a['A4_5layer_best_act_opt'])

Best among A1-A3: A1_3layer_relu_adam
Epoch 01/5 | train_loss=0.2925 train_acc=0.9098 | test_loss=0.1206 test_acc=0.9639
Epoch 02/5 | train_loss=0.1084 train_acc=0.9670 | test_loss=0.0881 test_acc=0.9725
Epoch 03/5 | train_loss=0.0731 train_acc=0.9768 | test_loss=0.0820 test_acc=0.9756
Epoch 04/5 | train_loss=0.0531 train_acc=0.9827 | test_loss=0.0841 test_acc=0.9748
Epoch 05/5 | train_loss=0.0448 train_acc=0.9858 | test_loss=0.0763 test_acc=0.9777
{'model': MLP(
  (net): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=64, bias=True)
    (7): ReLU()
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
), 'test_loss': 0.07625143861952238, 'test_accuracy': 0.9777, 'running_time_sec': 12.705849165999098}


### A.5
Summarize your observations in terms of choices of activation functions, optimizers, and model depth

In [7]:
# A.5: Observations summary for Task A
import pandas as pd

a_df = metrics.to_dataframe()
a_only = a_df[a_df['task'] == 'A'].sort_values('test_accuracy', ascending=False).copy()
display(a_only[['model', 'test_accuracy', 'running_time_sec', 'notes']])

best_a = a_only.iloc[0]
print('Task A best model:', best_a['model'])
print('Activation impact: ReLU generally converges faster and performed better than Sigmoid in this setup.')
print('Optimizer impact: Adam generally outperformed SGD on test accuracy with fewer tuning needs.')
print('Depth impact: Increasing depth can help, but gains may be small and come with extra runtime.')

,model,test_accuracy,running_time_sec,notes
0,A1_3layer_relu_adam,0.9785,10.610419,
3,A4_5layer_best_act_opt,0.9777,12.705849,"activation=relu, optimizer=adam"
1,A2_3layer_sigmoid_adam,0.9748,10.881141,
2,A3_3layer_relu_sgd,0.9309,10.104263,


Task A best model: A1_3layer_relu_adam
Activation impact: ReLU generally converges faster and performed better than Sigmoid in this setup.
Optimizer impact: Adam generally outperformed SGD on test accuracy with fewer tuning needs.
Depth impact: Increasing depth can help, but gains may be small and come with extra runtime.


## Task B: Convolutional Neural Networks

In [8]:
# Task B container
results_b = {}
print('Task B started: CNN experiments')

Task B started: CNN experiments


### B.1
Implement a four-layer CNN and evaluate its performance.

In [9]:
# B.1: Four-layer CNN
model_b1 = CNN4Layer()
opt_b1 = optim.Adam(model_b1.parameters(), lr=LEARNING_RATE)
results_b['B1_custom_4layer_cnn'] = train_model(
    model_b1, train_loader, test_loader, opt_b1, criterion, device, num_epochs=NUM_EPOCHS
)
metrics.add(
    'B',
    'B1_custom_4layer_cnn',
    results_b['B1_custom_4layer_cnn']['test_accuracy'],
    results_b['B1_custom_4layer_cnn']['running_time_sec'],
)
print(results_b['B1_custom_4layer_cnn'])

Epoch 01/5 | train_loss=0.1714 train_acc=0.9473 | test_loss=0.0505 test_acc=0.9839
Epoch 02/5 | train_loss=0.0485 train_acc=0.9849 | test_loss=0.0495 test_acc=0.9850
Epoch 03/5 | train_loss=0.0332 train_acc=0.9899 | test_loss=0.0299 test_acc=0.9897
Epoch 04/5 | train_loss=0.0230 train_acc=0.9927 | test_loss=0.0260 test_acc=0.9918
Epoch 05/5 | train_loss=0.0194 train_acc=0.9939 | test_loss=0.0309 test_acc=0.9898
{'model': CNN4Layer(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out

### B.2
Pick one CNN from the list {LeNet-5, AlexNet, VGG-16, GoogLeNet, ResNet-18}, and use it to train MNIST.\
Show the results and explain why you pick this one (hint: you do not need to train all of them; instead, you could study them online and pick one).

In [10]:
# B.2: LeNet-5 chosen from the allowed list
# Reason: LeNet-5 is lightweight, historically tied to digit recognition, and fits MNIST well.
model_b2 = LeNet5MNIST()
opt_b2 = optim.Adam(model_b2.parameters(), lr=LEARNING_RATE)
results_b['B2_lenet5'] = train_model(
    model_b2, train_loader, test_loader, opt_b2, criterion, device, num_epochs=NUM_EPOCHS
)
metrics.add(
    'B',
    'B2_lenet5',
    results_b['B2_lenet5']['test_accuracy'],
    results_b['B2_lenet5']['running_time_sec'],
    notes='Chosen because LeNet-5 is efficient and designed for digit-like image tasks',
)

import pandas as pd
b_df = metrics.to_dataframe()
display(b_df[b_df['task'] == 'B'].sort_values('test_accuracy', ascending=False)[['model', 'test_accuracy', 'running_time_sec', 'notes']])

Epoch 01/5 | train_loss=0.3471 train_acc=0.9027 | test_loss=0.1130 test_acc=0.9653
Epoch 02/5 | train_loss=0.0855 train_acc=0.9744 | test_loss=0.0684 test_acc=0.9788
Epoch 03/5 | train_loss=0.0569 train_acc=0.9826 | test_loss=0.0467 test_acc=0.9847
Epoch 04/5 | train_loss=0.0436 train_acc=0.9868 | test_loss=0.0413 test_acc=0.9872
Epoch 05/5 | train_loss=0.0358 train_acc=0.9890 | test_loss=0.0391 test_acc=0.9870


,model,test_accuracy,running_time_sec,notes
4,B1_custom_4layer_cnn,0.9918,67.143914,
5,B2_lenet5,0.9872,39.742813,Chosen because LeNet-5 is efficient and design...


## Task C. Performance Comparison and Analysis

In [ ]:
# Task C setup
assignment1_best_model_name = 'XGBoost'
assignment1_best_accuracy = 0.96950
assignment1_macro_f1 = 0.9693
assignment1_train_time_sec = 192.5334
assignment1_predict_time_sec = 0.0869
assignment1_running_time_sec = assignment1_train_time_sec
print(
    'Assignment 1 baseline set: '
    'XGBoost, accuracy=0.9695, macro_f1=0.9693, '
    'train=192.5334 sec, predict=0.0869 sec'
)

Assignment 1 baseline set: XGBoost, accuracy=0.9695, macro_f1=0.9693, train=192.5334 sec, predict=0.0869 sec


### C.1
Pick the best-performing MLP in Task A and the best-performing CNN in Task B. Compare them to the best performing model in YOUR OWN assignment 1.\
Which model performs the best?\
Summarize your findings and thoughts.

In [12]:
# C.1: Compare best Task A, best Task B, and Assignment 1 model
import pandas as pd

all_df = metrics.to_dataframe()

best_a_row = all_df[all_df['task'] == 'A'].sort_values('test_accuracy', ascending=False).head(1)
best_b_row = all_df[all_df['task'] == 'B'].sort_values('test_accuracy', ascending=False).head(1)

comparison_rows = []
if not best_a_row.empty:
    comparison_rows.append(best_a_row.iloc[0].to_dict())
if not best_b_row.empty:
    comparison_rows.append(best_b_row.iloc[0].to_dict())

comparison_rows.append({
    'task': 'Assignment1',
    'model': assignment1_best_model_name,
    'test_accuracy': assignment1_best_accuracy,
    'running_time_sec': assignment1_running_time_sec,
    'notes': 'From Assignment 1',
})

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df.sort_values('test_accuracy', ascending=False, na_position='last')[['task', 'model', 'test_accuracy', 'running_time_sec', 'notes']])

best_overall = comparison_df.dropna(subset=['test_accuracy']).sort_values('test_accuracy', ascending=False).head(1)
fastest_overall = comparison_df.dropna(subset=['running_time_sec']).sort_values('running_time_sec', ascending=True).head(1)

print('\nFinal Comparison Summary:')
if not best_overall.empty:
    print(f"1) Best test accuracy model: {best_overall.iloc[0]['model']} ({best_overall.iloc[0]['test_accuracy']:.4f})")
else:
    print('1) Best test accuracy model: update Assignment 1 accuracy to complete full comparison.')

if not fastest_overall.empty:
    print(f"2) Fastest model: {fastest_overall.iloc[0]['model']} ({fastest_overall.iloc[0]['running_time_sec']:.4f} sec)")
else:
    print('2) Fastest model: unavailable due to missing runtime values.')

print('3) Tradeoff: higher-capacity CNNs usually improve accuracy but need more runtime, while simpler MLPs train faster with somewhat lower peak accuracy.')

,task,model,test_accuracy,running_time_sec,notes
1,B,B1_custom_4layer_cnn,0.9918,67.143914,
0,A,A1_3layer_relu_adam,0.9785,10.610419,
2,Assignment1,XGBoost,0.9695,192.533400,From Assignment 1



Final Comparison Summary:
1) Best test accuracy model: B1_custom_4layer_cnn (0.9918)
2) Fastest model: A1_3layer_relu_adam (10.6104 sec)
3) Tradeoff: higher-capacity CNNs usually improve accuracy but need more runtime, while simpler MLPs train faster with somewhat lower peak accuracy.
